In [7]:
from torch.utils.data import TensorDataset
import torch

x, y = torch.rand(10, 2), torch.rand(10, 1)
dataset = TensorDataset(x, y)


# 1. DataLoader 

## What it is

The `DataLoader` is the piece that ties together the `Dataset` and the `Sampler`, and actually does the work of iterating and building ready-to-use batches for the model.

```
Dataset (answers "give me the item at index i")
    +
Sampler (decides which indices, and in what order)
    ↓
DataLoader (calls dataset[idx] repeatedly, groups results into batches, hands them over)
```

```python
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=32, shuffle=True)

for x_batch, y_batch in loader:
    ...
```

## What it does, step by step

1. Gets a sequence of indices from the sampler (e.g. `[7, 2, 9, 0, 5, 3, 8, 1, ...]`, shuffled or not — shuffling is the sampler's job, not the DataLoader's).
2. Regroups those indices into chunks of size `batch_size` (e.g. with `batch_size=4`: `[7,2,9,0]`, `[5,3,8,1]`, ...).
3. For each chunk, calls `dataset[idx]` once per index, collecting a list of individual items.
4. Packs (stacks) those individual items into a single batch tensor — this is the `collate_fn` step, `torch.stack` by default.
5. Yields that batch to you inside the `for` loop.

## Why it exists

`dataset[idx]` only returns **one** item. Models expect a whole **batch** at once (for GPU efficiency, among other reasons). The `DataLoader` is the bridge: it takes individual items and packs them into a single batched tensor, where the batch size becomes an extra leading dimension.

```
individual item shape (5,)       → batch shape (32, 5)
individual image shape (3,224,224) → batch shape (32,3,224,224)
```

The model then processes the whole batch in a single vectorized operation — there is no hidden loop processing item 0, then item 1, etc. Matrix multiplication (and other model ops) naturally operate on all rows of the batch at once.

## Manual equivalent (what the DataLoader automates)

```python
indices = [5, 12, 3, 47]                       # 4 indices, batch_size=4
items = [dataset[i] for i in indices]           # 4 (x, y) tuples
xs = torch.stack([item[0] for item in items])   # stack all x's
ys = torch.stack([item[1] for item in items])   # stack all y's
```

## Arguments

```python
DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
    sampler=None,
    batch_sampler=None,
    collate_fn=None,
    persistent_workers=False,
    prefetch_factor=2,
    generator=None,
)
```

| Argument | What it does |
|---|---|
| `dataset` | the `Dataset` (or `IterableDataset`) to pull items from |
| `batch_size` | how many items per batch |
| `shuffle` | shuffles the data each epoch. Internally creates a `RandomSampler` (or `SequentialSampler` if `False`). Only valid for map-style datasets. Mutually exclusive with `sampler` |
| `num_workers` | how many parallel worker processes load data. `0` = loading happens in the main process (no parallelism) |
| `pin_memory` | allocates batches in "pinned" (page-locked) memory, which speeds up the CPU → GPU transfer. Only useful if you're using a GPU |
| `drop_last` | if the last batch of an epoch is smaller than `batch_size`, drop it instead of keeping a smaller batch. Useful with layers sensitive to batch size (e.g. BatchNorm) |
| `sampler` | a custom `Sampler` controlling which indices and order to pull. Mutually exclusive with `shuffle` |
| `batch_sampler` | like `sampler`, but yields whole batches of indices at once instead of one index at a time. Mutually exclusive with `batch_size`, `shuffle`, `sampler`, `drop_last` |
| `collate_fn` | function that packs a list of individual items into one batch. Defaults to `default_collate` (stacks tensors). You write your own when items don't share the same shape (e.g. variable-length sequences needing padding) |
| `persistent_workers` | keeps worker processes alive between epochs instead of recreating them each time (avoids startup overhead). Only makes sense if `num_workers > 0` |
| `prefetch_factor` | how many batches each worker pre-loads ahead of time |
| `generator` | a `torch.Generator` to control the random seed used for shuffling, independent of the global seed |

## Practical notes

- `num_workers`: start around the number of physical CPU cores and tune from there. On notebooks/Windows, `num_workers > 0` can sometimes misbehave due to multiprocessing — test it.
- `pin_memory=True` only helps when data will be moved to a CUDA GPU afterward.
- `persistent_workers=True` is only meaningful when `num_workers > 0`.
- If GPU utilization is low during training, the bottleneck is usually data loading — increasing `num_workers` and enabling `pin_memory` are the first things to try.
- There is no family of different `DataLoader` classes in core PyTorch — it's a single class configured entirely through these arguments. (The experimental `DataLoader2` from the `torchdata` project exists but that project has been winding down, so the standard `DataLoader` remains the maintained path.)



### CPU vs GPU

Tensors returned by a `Dataset` live on the CPU by default (that's where any tensor is created unless you explicitly move it). The `Dataset`/`DataLoader` pipeline always operates on the CPU — you only move a batch to the GPU manually, inside the training loop, after it comes out of the `DataLoader`:

```python
for x_batch, y_batch in loader:
    x_batch = x_batch.to("cuda")   # moved to GPU only here, one batch at a time
    y_batch = y_batch.to("cuda")
```

You can always check where a tensor lives with `.device`:

```python
x.device   # → device(type='cpu')  or  device(type='cuda', index=0)
```

In [26]:
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

loader = DataLoader(
    dataset= dataset,
    batch_size = 2,
    drop_last = True,
    shuffle = True,
    pin_memory = True
)

for loadx, loady in loader:

    loadx, loady = loadx.to(device, non_blocking = True), loady.to(device, non_blocking = True)
    print(f'=' * 50)
    print(f'X: {loadx}')
    print(f'Y: {loady}')

X: tensor([[0.6047, 0.8611],
        [0.3488, 0.4123]], device='cuda:0')
Y: tensor([[0.0886],
        [0.4920]], device='cuda:0')
X: tensor([[0.7227, 0.3035],
        [0.8910, 0.1411]], device='cuda:0')
Y: tensor([[0.7722],
        [0.9249]], device='cuda:0')
X: tensor([[0.3315, 0.9323],
        [0.1358, 0.1352]], device='cuda:0')
Y: tensor([[0.9342],
        [0.4653]], device='cuda:0')
X: tensor([[0.8069, 0.1732],
        [0.7014, 0.6585]], device='cuda:0')
Y: tensor([[0.1576],
        [0.5384]], device='cuda:0')
X: tensor([[0.2943, 0.1182],
        [0.3237, 0.6493]], device='cuda:0')
Y: tensor([[0.6521],
        [0.3729]], device='cuda:0')
